# ServiceNow Knowledge Connector Provisioning Helper

This notebook helps you provision the Microsoft 365 Copilot ServiceNow Knowledge connector using OAuth 2.0.

It automates what is safe and repeatable:
* load and validate environment settings from .env
* verify ServiceNow REST connectivity and Knowledge table read access
* probe ServiceNow API metadata to see if OAuth provisioning through API is feasible
* generate a deployment packet for Microsoft 365 admin center

Current limitation:
* connector creation in Microsoft 365 admin center is still a manual UI step in this workflow
* ServiceNow OAuth provisioning through API varies by tenant/version, so this notebook probes capabilities before any write operation

## 1) Imports

In [ ]:
import json
import os
from pathlib import Path

import requests
from dotenv import dotenv_values, find_dotenv, load_dotenv

print('Imports loaded.')

## 2) Load .env and helper functions

In [ ]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    env_path = str(Path.cwd().parent / '.env') if (Path.cwd().parent / '.env').exists() else str(Path.cwd() / '.env')

load_dotenv(env_path, override=True)
env_values = dotenv_values(env_path)

def env(name, default=''):
    value = os.getenv(name, default)
    return value.strip() if isinstance(value, str) else value

def mask(value):
    if not value:
        return ''
    if len(value) <= 6:
        return '***'
    return value[:3] + '***' + value[-3:]

print(f'Loaded environment file: {env_path}')

## 3) Read connector configuration

In [ ]:
config = {
    'environment_name': env('COPILOT_SN_ENVIRONMENT_NAME'),
    'display_name': env('COPILOT_SN_CONNECTOR_DISPLAY_NAME'),
    'instance_url': env('COPILOT_SN_INSTANCE_URL'),
    'flow_mode': env('COPILOT_SN_FLOW_MODE', 'Simple'),
    'api_namespace': env('COPILOT_SN_API_NAMESPACE'),
    'query_string': env('COPILOT_SN_QUERY_STRING', 'active=true^workflow_state=published'),
    'access_permissions': env('COPILOT_SN_ACCESS_PERMISSIONS', 'DataSource'),
    'map_identities_formula': env('COPILOT_SN_MAP_IDENTITIES_FORMULA'),
    'limited_rollout': env('COPILOT_SN_LIMITED_ROLLOUT', 'false'),
    'rollout_users': env('COPILOT_SN_ROLLOUT_USERS'),
    'rollout_groups': env('COPILOT_SN_ROLLOUT_GROUPS'),
    'oauth_client_id': env('COPILOT_SN_OAUTH_CLIENT_ID'),
    'oauth_client_secret': env('COPILOT_SN_OAUTH_CLIENT_SECRET'),
    'oauth_redirect_url': env('COPILOT_SN_OAUTH_REDIRECT_URL', 'https://gcs.office.com/v1.0/admin/oauth/callback'),
    'oauth_auth_scope': env('COPILOT_SN_OAUTH_AUTH_SCOPE', 'useraccount'),
    'oauth_refresh_token_lifespan': env('COPILOT_SN_OAUTH_REFRESH_TOKEN_LIFESPAN', '31536000'),
    'oauth_access_token_lifespan': env('COPILOT_SN_OAUTH_ACCESS_TOKEN_LIFESPAN', '43200'),
    'crawl_username': env('COPILOT_SN_CRAWL_USERNAME'),
    'crawl_password': env('COPILOT_SN_CRAWL_PASSWORD'),
}

safe_view = dict(config)
safe_view['oauth_client_secret'] = mask(safe_view['oauth_client_secret'])
safe_view['crawl_password'] = mask(safe_view['crawl_password'])
print(json.dumps(safe_view, indent=2))

## 4) Validate required values

In [ ]:
required = [
    'display_name',
    'instance_url',
    'oauth_client_id',
    'oauth_client_secret',
    'crawl_username',
    'crawl_password',
]

missing = [name for name in required if not config.get(name)]

if missing:
    print('Missing required values:')
    for name in missing:
        print(f'- {name}')
    raise ValueError('Populate missing values in .env before continuing.')

if not config['instance_url'].startswith('https://'):
    raise ValueError('COPILOT_SN_INSTANCE_URL must start with https://')

if config['flow_mode'] not in {'Simple', 'Advanced'}:
    raise ValueError('COPILOT_SN_FLOW_MODE must be Simple or Advanced')

if config['flow_mode'] == 'Advanced' and not config['api_namespace']:
    raise ValueError('COPILOT_SN_API_NAMESPACE is required when flow mode is Advanced')

print('Validation passed.')

## 5) ServiceNow connectivity and read-access preflight

This checks if the crawl account can call REST APIs and read knowledge content metadata.

In [ ]:
session = requests.Session()
session.auth = (config['crawl_username'], config['crawl_password'])
session.headers.update({'Accept': 'application/json'})

def get_json(path, params=None):
    url = config['instance_url'].rstrip('/') + path
    response = session.get(url, params=params, timeout=60)
    return response

checks = {}

resp_ping = get_json('/api/now/table/sys_user', params={'sysparm_limit': 1, 'sysparm_fields': 'sys_id'})
checks['rest_ping_status'] = resp_ping.status_code

resp_kb = get_json('/api/now/table/kb_knowledge', params={'sysparm_limit': 1, 'sysparm_fields': 'sys_id,number,short_description'})
checks['kb_knowledge_status'] = resp_kb.status_code
if resp_kb.ok:
    data = resp_kb.json()
    checks['kb_knowledge_row_count'] = len(data.get('result', []))

print(json.dumps(checks, indent=2))

if resp_ping.status_code >= 400:
    raise RuntimeError('ServiceNow REST ping failed. Check instance URL and crawl credentials.')

if resp_kb.status_code >= 400:
    raise RuntimeError('kb_knowledge read failed. Validate ServiceNow roles/ACLs for the crawl account.')

## 6) OAuth provisioning capability probe

I do not know a single stable ServiceNow API contract that is guaranteed across all versions/tenants for creating OAuth app registrations for this connector.

This probe discovers candidate OAuth/OIDC tables in your instance so you can decide whether to automate writes in a later step.

In [ ]:
probe_params = {
    'sysparm_query': 'nameLIKEoauth^ORnameLIKEoidc',
    'sysparm_limit': 200,
    'sysparm_fields': 'name,label,super_class'
}
resp_tables = get_json('/api/now/table/sys_db_object', params=probe_params)

probe_result = {
    'status_code': resp_tables.status_code,
    'table_count': 0,
    'sample_tables': []
}

if resp_tables.ok:
    rows = resp_tables.json().get('result', [])
    names = sorted({row.get('name', '') for row in rows if row.get('name')})
    probe_result['table_count'] = len(names)
    probe_result['sample_tables'] = names[:30]

print(json.dumps(probe_result, indent=2))

if not resp_tables.ok:
    print('Could not query sys_db_object. This usually means restricted metadata access.')

## 7) Provision OAuth 2.0 client in ServiceNow (create or reuse)

This cell creates a ServiceNow OAuth client record if one does not exist yet.

Behavior:
- checks for an existing oauth_entity by name
- reuses existing credentials when present
- otherwise creates a new oauth_entity with connector-friendly defaults
- updates in-memory config with client ID and secret
- optionally writes client ID and secret back to .env

In [ ]:
import secrets
from dotenv import set_key

write_back_to_env = True
oauth_entity_name = f"Microsoft Search OAuth - {config.get('environment_name') or 'env'}"

def post_json(path, payload):
    url = config['instance_url'].rstrip('/') + path
    response = session.post(url, json=payload, timeout=60)
    return response

lookup_resp = get_json(
    '/api/now/table/oauth_entity',
    params={
        'sysparm_query': f"name={oauth_entity_name}",
        'sysparm_limit': 1,
        'sysparm_fields': 'sys_id,name,client_id,client_secret,redirect_url,inbound_grant_type,default_grant_type,active,scope_restriction_status',
    },
)

if lookup_resp.status_code >= 400:
    raise RuntimeError(
        f"OAuth lookup failed: {lookup_resp.status_code} {lookup_resp.text}"
    )

lookup_rows = lookup_resp.json().get('result', [])
created_new = False

if lookup_rows:
    oauth_entity = lookup_rows[0]
else:
    generated_client_id = secrets.token_hex(16)
    generated_client_secret = secrets.token_urlsafe(32)

    create_payload = {
        'name': oauth_entity_name,
        'type': 'client',
        'active': 'true',
        'client_id': generated_client_id,
        'client_secret': generated_client_secret,
        'redirect_url': config['oauth_redirect_url'],
        'inbound_grant_type': 'authz_code',
        'default_grant_type': 'authorization_code',
        'token_format': 'opaque',
        'scope_restriction_status': 'unrestricted',
        'access_token_lifespan': config['oauth_access_token_lifespan'],
        'refresh_token_lifespan': config['oauth_refresh_token_lifespan'],
        'auth_code_lifespan': '60',
        'id_token_lifespan': '86400',
        'message_signature_lifespan': '60',
        'certificate_url': '',
        'salt': '',
    }

    create_resp = post_json(
        '/api/now/table/oauth_entity?sysparm_fields=sys_id,name,client_id,client_secret,redirect_url,inbound_grant_type,default_grant_type,active',
        create_payload,
    )

    if create_resp.status_code >= 400:
        raise RuntimeError(
            f"OAuth create failed: {create_resp.status_code} {create_resp.text}"
        )

    oauth_entity = create_resp.json().get('result', {})
    created_new = True

config['oauth_client_id'] = oauth_entity.get('client_id', config['oauth_client_id'])
config['oauth_client_secret'] = oauth_entity.get('client_secret', config['oauth_client_secret'])

# Warn if existing entity has the wrong scope restriction setting.
# The Table API ACL blocks PATCH on oauth_entity, so this must be fixed manually in the UI.
scope_status = oauth_entity.get('scope_restriction_status', '')
scope_warning = None
if not created_new and scope_status != 'unrestricted':
    scope_warning = (
        f"WARNING: OAuth entity scope_restriction_status = '{scope_status}'. "
        "Tokens issued by this app cannot call the ServiceNow Table API. "
        "Fix in ServiceNow UI: System OAuth > Application Registry > "
        f"'{oauth_entity_name}' > set 'Scope restriction' to Unrestricted > Save."
    )
    print(scope_warning)

if write_back_to_env:
    set_key(env_path, 'COPILOT_SN_OAUTH_CLIENT_ID', config['oauth_client_id'])
    set_key(env_path, 'COPILOT_SN_OAUTH_CLIENT_SECRET', config['oauth_client_secret'])

summary = {
    'oauth_entity_name': oauth_entity.get('name'),
    'created_new': created_new,
    'scope_restriction_status': scope_status or 'unrestricted',
    'scope_warning': scope_warning,
    'oauth_client_id': config['oauth_client_id'],
    'oauth_client_secret': mask(config['oauth_client_secret']),
    'wrote_to_env': write_back_to_env,
    'env_path': env_path,
}

print(json.dumps(summary, indent=2))

## 8) Build wizard-ready setup guide for Microsoft 365 admin center

This step renders a user-friendly guide that follows the connector wizard tabs:
- Setup
- Users
- Content
- Sync

Use it as a copy/paste checklist while you configure the connector UI.

In [ ]:
from IPython.display import Markdown, display

def to_bool(value):
    return str(value).strip().lower() in {'1', 'true', 'yes', 'y'}

def csv_items(value):
    if not value:
        return []
    return [item.strip() for item in str(value).split(',') if item.strip()]

deployment_packet = {
    'connector': {
        'display_name': config['display_name'],
        'instance_url': config['instance_url'],
        'flow_mode': config['flow_mode'],
        'api_namespace': config['api_namespace'] if config['flow_mode'] == 'Advanced' else '',
        'authentication_type': 'OAuth2',
    },
    'oauth2': {
        'client_id': config['oauth_client_id'],
        'client_secret': '<hidden>',
        'redirect_url': config['oauth_redirect_url'],
        'auth_scope': config['oauth_auth_scope'],
        'refresh_token_lifespan_seconds': config['oauth_refresh_token_lifespan'],
        'access_token_lifespan_seconds': config['oauth_access_token_lifespan'],
    },
    'crawl_account': {
        'username': config['crawl_username'],
        'password': '<hidden>'
    },
    'content': {
        'query_string': config['query_string'],
        'access_permissions': config['access_permissions'],
        'map_identities_formula': config['map_identities_formula'],
    },
    'rollout': {
        'limited_rollout': config['limited_rollout'],
        'users_csv': config['rollout_users'],
        'groups_csv': config['rollout_groups'],
    }
}

users_rollout = csv_items(deployment_packet['rollout']['users_csv'])
groups_rollout = csv_items(deployment_packet['rollout']['groups_csv'])
limited_rollout = to_bool(deployment_packet['rollout']['limited_rollout'])

setup_rows = [
    ('Display name', deployment_packet['connector']['display_name']),
    ('User criteria flow', deployment_packet['connector']['flow_mode']),
    ('API namespace', deployment_packet['connector']['api_namespace'] or '(leave blank for Simple flow)'),
    ('ServiceNow URL', deployment_packet['connector']['instance_url']),
    ('Authentication type', deployment_packet['connector']['authentication_type']),
    ('OAuth Client ID', deployment_packet['oauth2']['client_id']),
    ('OAuth Client Secret', 'Use COPILOT_SN_OAUTH_CLIENT_SECRET from .env'),
    ('OAuth Redirect URL', deployment_packet['oauth2']['redirect_url']),
    ('OAuth Auth Scope', deployment_packet['oauth2']['auth_scope']),
    ('Rollout to limited audience', 'Yes' if limited_rollout else 'No'),
]

users_rows = [
    ('Access permissions', 'Only people with access to this data source' if deployment_packet['content']['access_permissions'] == 'DataSource' else 'Everyone'),
    ('Identity mapping', deployment_packet['content']['map_identities_formula'] or 'Default Microsoft Entra mapping (UPN/Mail)'),
]

content_rows = [
    ('Query string', deployment_packet['content']['query_string']),
    ('Manage properties', 'Use defaults unless you need custom schema mapping'),
]

sync_rows = [
    ('Incremental crawl', 'Every 15 minutes (default/recommended)'),
    ('Full crawl', 'Every 1 day (default/recommended)'),
]

def markdown_table(title, rows):
    header = f"### {title}\n| Wizard field | Value to use |\n|---|---|\n"
    body = '\n'.join([f"| {label} | {value} |" for label, value in rows])
    return header + body

rollout_notes = [
    f"* Limited rollout: {'enabled' if limited_rollout else 'disabled'}",
    f"* Rollout users count: {len(users_rollout)}",
    f"* Rollout groups count: {len(groups_rollout)}",
]

if users_rollout:
    rollout_notes.append('* Users: ' + ', '.join(users_rollout))
if groups_rollout:
    rollout_notes.append('* Groups: ' + ', '.join(groups_rollout))

troubleshooting = [
    '* If the Users tab only allows Everyone, the ServiceNow crawl account is missing access to one or more user criteria related tables.',
    '* Re-run Step 5 preflight after role/table permission changes in ServiceNow.',
]

guide_md = '\n\n'.join([
    '## Connector Wizard Guide (copy/paste by tab)',
    markdown_table('Setup tab', setup_rows),
    markdown_table('Users tab', users_rows),
    markdown_table('Content tab', content_rows),
    markdown_table('Sync tab', sync_rows),
    '### Rollout summary\n' + '\n'.join(rollout_notes),
    '### Troubleshooting tips\n' + '\n'.join(troubleshooting),
])

display(Markdown(guide_md))

print('Raw packet is still available as deployment_packet if you need JSON automation output.')

## 9) Manual steps that are still required

1. **Verify OAuth scope restriction** (critical): In ServiceNow, go to System OAuth > Application Registry > open the OAuth entity created above. Confirm the 'Scope restriction' field is set to **Unrestricted**. If it shows 'Access allowed only to APIs in selected scope', change it to Unrestricted and click Save. If this is wrong, the connector Authorize step will fail with 'missing access to kb_knowledge'.
2. In Microsoft 365 admin center, create the ServiceNow Knowledge connector using this notebook's deployment packet.
3. In connector custom setup, confirm query string and access permissions.
4. Start crawl and monitor first full crawl completion.
5. Validate by checking known article sys_id in indexed content diagnostics.
6. If your org requires it, review OAuth settings in ServiceNow Application Registry and rotate secret on your standard cadence.

In [ ]:
admin_center_url = 'https://admin.microsoft.com/Adminportal/Home#/copilot/connectors'
print('Open this URL to create the connector:')
print(admin_center_url)